In [ ]:
import os

!pip3 install -q virtualenv
!virtualenv /content/lego_env --python=python3.12

!"/content/lego_env/bin/pip" install -q \
torch torchvision --index-url https://download.pytorch.org/whl/cu121

!"/content/lego_env/bin/pip" install -q \
timm==0.9.16 \
    ultralytics \
    pytorch-metric-learning \
    Pillow \
    tqdm \
    PyYAML \
    scipy

# Write verify script to file
with open('/content/verify.py', 'w') as f:
    f.write("""
import torch, timm, scipy
from pytorch_metric_learning.losses import ArcFaceLoss
print(f'torch: {torch.__version__}')
print(f'scipy: {scipy.__version__}')
print(f'timm: {timm.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
print('All imports OK')
""")

!"/content/lego_env/bin/python" /content/verify.py

In [ ]:
from google.colab import drive
import os
import sys

drive.mount('/content/drive', force_remount=True)

BASE_PATH = "/content/drive/MyDrive/Brick_Detection/"
LOCAL_DATA = "/content/data/"

os.makedirs(LOCAL_DATA, exist_ok=True)
REPO_DIR = os.path.join(BASE_PATH, "repo")

if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    !git clone https://github.com/Crightub/lego-brick-detection.git {REPO_DIR}
else:
    print("Repo already exists, pulling latest changes...")
    !git -C {REPO_DIR} pull

In [ ]:
print("Copying zips to local storage...")
!cp {BASE_PATH}repo/data/train/crops_train.zip /content/crops_train.zip
!cp {BASE_PATH}repo/data/val/crops_val.zip /content/crops_val.zip
print("Copy done.")

print("Unzipping train crops...")
!unzip -q /content/crops_train.zip -d {LOCAL_DATA}

print("Unzipping val crops...")
!unzip -q /content/crops_val.zip -d {LOCAL_DATA}

print("Done! Verifying structure...")
!find /content/data/train/crops -maxdepth 1 -type d | wc -l
!find /content/data/val/crops -maxdepth 1 -type d | wc -l

In [ ]:
import os

REPO_DIR = "/content/drive/MyDrive/Brick_Detection/repo"
BASE_PATH = "/content/drive/MyDrive/Brick_Detection/"
PYTHON = "/content/lego_env/bin/python"

# Write the config to a small launcher script so you don't need argparse over CLI
launcher = f"""
import sys
sys.path.insert(0, '{REPO_DIR}')

import argparse
from ViTS16_Stage2.train import main

preset = argparse.Namespace(
    crops_train_dir='/content/data/train/crops',
    crops_val_dir='/content/data/val/crops',
    lr=1e-4,
    weight_decay=1e-4,
    batch_size=64,
    epochs=30,
    milestones=[15, 25],
    gamma=0.1,
    device='0',
    model_out_dir='{BASE_PATH}model/stage2',
    resume=False,
    start_epoch=0,
)

main(preset)
"""

with open('/content/run_stage2.py', 'w') as f:
    f.write(launcher)

# Run using the clean virtualenv Python — completely isolated from Colab's scipy
!{PYTHON} /content/run_stage2.py